In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("../cleanData.csv")

###  One-Hot Encoding for Month

In [3]:
month_dummies = pd.get_dummies(df['Month'], prefix='Month', drop_first=False)
df = pd.concat([df, month_dummies], axis=1)
df = df.drop(columns=['Month', 'Date', 'Year'])
df.drop(columns=['Month_sin', 'Month_cos'], inplace=True)

# PCA

In [5]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [7]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [15]:
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

# OpTuna

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [10]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(score_func=f_regression,
        #                          k=trial.suggest_int("k_features", 5, X.shape[1]))),
        ('pca', PCA(
            n_components= 0.95 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-04 17:13:44,815] A new study created in memory with name: no-name-e9e855c5-3d92-4eaf-90fe-3f169a180a47
[I 2026-05-04 17:13:47,609] Trial 0 finished with value: 0.24829540432519395 and parameters: {'n_estimators': 205, 'max_depth': 23, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.24829540432519395.
[I 2026-05-04 17:13:49,262] Trial 1 finished with value: 0.2063410908737977 and parameters: {'n_estimators': 78, 'max_depth': 35, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.24829540432519395.
[I 2026-05-04 17:13:52,354] Trial 2 finished with value: 0.26187122975907684 and parameters: {'n_estimators': 257, 'max_depth': 29, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.26187122975907684.
[I 2026-05-04 17:13:52,758] Trial 3 finished with value: 0.15473045964471796 and parameters: {'n_estimators': 57, 'max_depth': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 2 with va

========== RF BEST ==========
{'n_estimators': 272, 'max_depth': 23, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.2804


In [11]:
def xgb_objective(trial):

    model = xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(
        #     score_func=f_regression,
        #     k=trial.suggest_int("k_features", 5, X.shape[1])
        # )),
        ('pca', PCA(
            n_components= 0.95 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")

xgb_study.optimize(xgb_objective, n_trials=50)

print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-04 17:15:06,090] A new study created in memory with name: no-name-a79ec02c-1ece-4c07-9971-3ee159e6515f
[I 2026-05-04 17:15:18,459] Trial 0 finished with value: 0.2749224267713565 and parameters: {'n_estimators': 257, 'learning_rate': 0.04755949360582842, 'max_depth': 14, 'subsample': 0.803039106841018, 'colsample_bytree': 0.8938183436204001}. Best is trial 0 with value: 0.2749224267713565.
[I 2026-05-04 17:15:19,916] Trial 1 finished with value: 0.26500804227848274 and parameters: {'n_estimators': 230, 'learning_rate': 0.023687117291293174, 'max_depth': 6, 'subsample': 0.8523885432952515, 'colsample_bytree': 0.7747629291820372}. Best is trial 0 with value: 0.2749224267713565.
[I 2026-05-04 17:15:21,698] Trial 2 finished with value: 0.26158797955648205 and parameters: {'n_estimators': 99, 'learning_rate': 0.09239676519611228, 'max_depth': 9, 'subsample': 0.6963750966338692, 'colsample_bytree': 0.6889201048230638}. Best is trial 0 with value: 0.2749224267713565.
[I 2026-05-04 

========== XGB BEST ==========
{'n_estimators': 399, 'learning_rate': 0.010144257857367212, 'max_depth': 13, 'subsample': 0.9347677972531437, 'colsample_bytree': 0.5568470263921705}
Best R²: 0.3010


# Start

In [16]:
xgb_model = xgb.XGBRegressor(
    n_estimators=399, 
    learning_rate=0.01, 
    max_depth=13, 
    subsample=0.66, 
    colsample_bytree=0.93, 
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=272, 
    max_depth=23, 
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1, 
    random_state=42
)
# {'n_estimators': 272, 'max_depth': 23, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

In [17]:
voting = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.3619
RMSE: 6.6750


# K-Fold

In [18]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# %%
# K-Fold Validation 

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components= 0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Train
    voting      = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
    voting.fit(X_train_pca, y_train)

    # Evaluate
    preds = voting.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.3918  |  RMSE: 7.3095
Fold 2  →  R²: 0.3824  |  RMSE: 8.5464
Fold 3  →  R²: 0.2139  |  RMSE: 8.6338
Fold 4  →  R²: 0.3619  |  RMSE: 6.6750

--- 4-Fold Summary ---
R²   mean: 0.3375  ±  0.0722
RMSE mean: 7.7912  ±  0.8304
